# Week 7 - Notebook 4: Evaluation

## Goal
Evaluate the fine-tuned model:
1. Load fine-tuned model from HuggingFace Hub
2. Run on test set
3. Generate visualizations
4. Compare with baseline

Expected result: ~$40 error (vs $110 baseline)

## Time: 15-20 minutes

In [ ]:
import sys
sys.path.append('..')

import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from src.items import Item
from src.evaluator import evaluate
from src.config import config

# Load environment
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")

## Configuration

In [ ]:
# Model to evaluate
FINETUNED_MODEL_ID = "your-username/llama-pricer-lite"  # Replace with your model

print(f"Will evaluate model: {FINETUNED_MODEL_ID}")
config.display()

## Load Test Data

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")

## Load Fine-tuned Model

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA adapters from: {FINETUNED_MODEL_ID}")
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_ID)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Fine-tuned model loaded")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Test on Sample Products

In [ ]:
def predict_finetuned(item: Item) -> str:
    """
    Predict price using fine-tuned model
    """
    # Create prompt
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)
    
    prompt = item.test_prompt()
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract completion
    completion = response.split(config.PREFIX)[-1].strip()
    
    return completion

In [ ]:
# Test on a few examples
print("Testing fine-tuned model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    prediction = predict_finetuned(item)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)

## Full Evaluation on Test Set

In [ ]:
# Evaluate on full test set
results = evaluate(
    predict_finetuned,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("FINE-TUNED MODEL RESULTS")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")

## Comparison with Baseline

In [ ]:
import plotly.graph_objects as go

# Expected results (update with your actual results)
baseline_error = 110.72  # Base Llama
finetuned_error = results['average_error']
improvement = (baseline_error - finetuned_error) / baseline_error * 100

# Create comparison chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=["Base Llama 3.2", "Fine-tuned Llama"],
    y=[baseline_error, finetuned_error],
    marker_color=["darkred", "green"],
    text=[f"${baseline_error:.2f}", f"${finetuned_error:.2f}"],
    textposition="outside",
))

fig.update_layout(
    title=f"Fine-tuning Improvement: {improvement:.1f}% reduction in error",
    yaxis_title="Mean Absolute Error ($)",
    width=800,
    height=500,
)

fig.show()

print(f"\n🎉 Fine-tuning reduced error by {improvement:.1f}%!")
print(f"   From ${baseline_error:.2f} → ${finetuned_error:.2f}")

## Compare with All Models

In [ ]:
# All model results (from Week 7 curriculum)
all_results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", 110.72),
    ("Fine-tuned (Your Model)", "green", finetuned_error),
]

labels, colors, values = zip(*all_results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="All Models Comparison - Price Prediction Error",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1200,
    height=600,
)

fig.show()

## Summary

✅ Evaluation complete!

**Your Results:**
- Fine-tuned model: $XX.XX error
- Improvement over base: XX%
- Comparison with GPT-5.1: [better/worse] by $XX

**Expected Results:**
- Lite mode: ~$65 error (good)
- Full mode: ~$40 error (excellent - beats GPT-5.1!)

**Key Achievements:**
1. ✅ Successfully fine-tuned Llama 3.2 with QLoRA
2. ✅ Reduced error by 60-70% vs base model
3. ✅ Achieved competitive/better performance than frontier models
4. ✅ No API costs - can run locally or on cheap GPUs
5. ✅ Full control over model and data

**Next Steps:**
- Deploy to production (Modal.com, HuggingFace Endpoints, etc.)
- Try advanced techniques (multi-domain adapters, reasoning chains)
- Move to Week 8 for multi-agent systems
- Combine Week 7 + Week 8 for complete platform